In [10]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

In [11]:
train_poc = pd.read_csv(
    DATA_DIR / "train_poc.csv",
    usecols=["id", "input_text", "true_risks"]
)

train_input = pd.read_csv(
    OUTPUT_DIR / "stage_c_train_input.csv"
)

test_input = pd.read_csv(
    OUTPUT_DIR / "stage_c_test_input.csv"
)


train_poc["true_risks"] = (
    train_poc["true_risks"].apply(ast.literal_eval)
)

In [12]:
json_columns = [
    "stage_a_predictions",
    "stage_b_predictions",
    "accepted_predictions",
    "stage_b_probabilities"
]


for dataframe in [train_input, test_input]:

    for column in json_columns:
        dataframe[column] = dataframe[column].apply(
            json.loads
        )

    dataframe["route_to_stage_c"] = (
        dataframe["route_to_stage_c"]
        .astype(str)
        .str.lower()
        .eq("true")
    )

In [13]:
saved_embeddings = np.load(
    OUTPUT_DIR / "stage_b_embeddings.npz",
    allow_pickle=False
)

train_ids = saved_embeddings["train_ids"]
test_ids = saved_embeddings["test_ids"]

train_embeddings = saved_embeddings["train_embeddings"]
test_embeddings = saved_embeddings["test_embeddings"]


assert np.array_equal(
    train_ids,
    train_poc["id"].to_numpy()
)

print(
    "Embedding model:",
    saved_embeddings["model_name"].item()
)

Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [14]:
routed_train = train_input[
    train_input["route_to_stage_c"]
].copy()

routed_test = test_input[
    test_input["route_to_stage_c"]
].copy()


sample_sizes = {
    "predictions_disagree": 45,
    "both_stages_empty": 40,
    "stage_b_empty": 15
}


training_samples = []

for reason, sample_size in sample_sizes.items():

    group = routed_train[
        routed_train["stage_c_route_reason"] == reason
    ]

    training_samples.append(
        group.sample(
            n=min(sample_size, len(group)),
            random_state=42
        )
    )


train_llm_input = pd.concat(
    training_samples,
    ignore_index=True
)

test_llm_input = routed_test.reset_index(
    drop=True
)


print("Training LLM records:", len(train_llm_input))
print("Test LLM records:", len(test_llm_input))

Training LLM records: 100
Test LLM records: 194


In [15]:
train_position = {
    event_id: position
    for position, event_id in enumerate(train_ids)
}

test_position = {
    event_id: position
    for position, event_id in enumerate(test_ids)
}


train_query_embeddings = train_embeddings[
    [
        train_position[event_id]
        for event_id in train_llm_input["id"]
    ]
]

test_query_embeddings = test_embeddings[
    [
        test_position[event_id]
        for event_id in test_llm_input["id"]
    ]
]

In [16]:
train_lookup = train_poc.set_index("id")


def add_retrieved_examples(
    queries,
    query_embeddings,
    exclude_self=False,
    k=3
):

    similarities = (
        query_embeddings
        @ train_embeddings.T
    )

    if exclude_self:

        for row_number, event_id in enumerate(
            queries["id"]
        ):
            similarities[
                row_number,
                train_position[event_id]
            ] = -np.inf

    nearest_positions = np.argsort(
        similarities,
        axis=1
    )[:, -k:][:, ::-1]

    output = queries.copy()

    output["retrieved_examples"] = [
        [
            {
                "id": int(train_ids[position]),
                "text": train_lookup.at[
                    int(train_ids[position]),
                    "input_text"
                ],
                "true_risks": train_lookup.at[
                    int(train_ids[position]),
                    "true_risks"
                ],
                "similarity": round(
                    float(similarities[row_number, position]),
                    4
                )
            }
            for position in positions
        ]
        for row_number, positions in enumerate(
            nearest_positions
        )
    ]

    return output

In [17]:
train_llm_input = add_retrieved_examples(
    train_llm_input,
    train_query_embeddings,
    exclude_self=True
)

test_llm_input = add_retrieved_examples(
    test_llm_input,
    test_query_embeddings
)


train_llm_input[
    [
        "id",
        "stage_c_route_reason",
        "retrieved_examples"
    ]
].head()

,id,stage_c_route_reason,retrieved_examples
0,2766,predictions_disagree,"[{'id': 2765, 'text': 'UPDATE: Average waiting..."
1,4557,predictions_disagree,"[{'id': 3872, 'text': 'Operations at Port of S..."
2,2574,predictions_disagree,"[{'id': 1718, 'text': 'Japan: Around 1.8 milli..."
3,2911,predictions_disagree,"[{'id': 2830, 'text': 'UPDATE: Intermittent po..."
4,38,predictions_disagree,"[{'id': 798, 'text': 'UPDATE - USA, New York: ..."


In [18]:
%pip install -q -U "transformers>=4.37.0" accelerate

In [19]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [20]:
import torch

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer


QWEN_MODEL_NAME = (
    "Qwen/Qwen2.5-3B-Instruct"
)


tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Qwen loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully.


In [21]:
messages = [
    {
        "role": "user",
        "content": "Reply exactly with: ready"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():

    generated = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    generated[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

ready


In [ ]:
# %pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import os
# import getpass

# from openai import OpenAI


# api_key = os.getenv("OPENAI_API_KEY")

# if not api_key:
#     api_key = getpass.getpass(
#         "Enter OpenAI API key: "
#     )

# client = OpenAI(api_key=api_key)

# LLM_MODEL = "gpt-4.1-mini"

In [24]:
RISK_TAXONOMY = {
    "weather_disruption":
        "Disruption caused by severe weather.",

    "natural_disaster":
        "Disruption caused by earthquakes, tsunamis, volcanoes or landslides.",

    "port_operational_disruption":
        "Port congestion, delays, backlogs or reduced operational capacity.",

    "port_closure":
        "Full or partial closure or suspension of a port or terminal.",

    "labor_strike_disruption":
        "Disruption caused by strikes or other labor action.",

    "maritime_security_navigation_disruption":
        "Piracy, maritime security threats, navigation restrictions or waterway disruption."
}


RISK_IDS = list(RISK_TAXONOMY)


QWEN_SYSTEM_PROMPT = """
You are the final adjudicator for a multi-label maritime risk classifier.

Risk taxonomy:
""" + json.dumps(
    RISK_TAXONOMY,
    indent=2
) + """

Use the target event text as the main evidence.
Stage A, Stage B and retrieved examples are supporting evidence.

Return only valid JSON in this exact format:
{
  "predicted_risks": ["risk_id"],
  "reason": "short explanation"
}

Use only risk IDs from the taxonomy.
"""

In [25]:
def simple_fallback(row):

    if row["stage_b_predictions"]:
        return row["stage_b_predictions"]

    if row["stage_a_predictions"]:
        return row["stage_a_predictions"]

    return []


def generate_with_qwen(user_prompt):

    messages = [
        {
            "role": "system",
            "content": QWEN_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        generated = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

In [26]:
def parse_qwen_response(text):

    cleaned = (
        text.replace("```json", "")
        .replace("```", "")
        .strip()
    )

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found.")

    result = json.loads(
        cleaned[start:end + 1]
    )

    predictions = result["predicted_risks"]

    if not isinstance(predictions, list):
        raise ValueError("predicted_risks is not a list.")

    if any(
        risk not in RISK_IDS
        for risk in predictions
    ):
        raise ValueError("Invalid risk returned.")

    return (
        list(dict.fromkeys(predictions)),
        str(result["reason"])
    )


def adjudicate(row):

    case = {
        "event_text": row["input_text"],
        "stage_a_predictions":
            row["stage_a_predictions"],
        "stage_b_predictions":
            row["stage_b_predictions"],
        "stage_b_probabilities":
            row["stage_b_probabilities"],
        "route_reason":
            row["stage_c_route_reason"],
        "retrieved_examples":
            row["retrieved_examples"]
    }

    last_error = ""

    for _ in range(2):

        try:
            response_text = generate_with_qwen(
                json.dumps(
                    case,
                    ensure_ascii=False
                )
            )

            predictions, reason = parse_qwen_response(
                response_text
            )

            return predictions, reason, False

        except Exception as error:
            last_error = str(error)

    return (
        simple_fallback(row),
        f"Fallback used: {last_error[:150]}",
        True
    )

In [27]:
test_prediction = adjudicate(
    train_llm_input.iloc[0]
)

print(test_prediction)

(['port_operational_disruption', 'weather_disruption'], 'The event mentions decreased waiting times and potential disruption due to a typhoon, aligning with port operational disruption and weather disruption risks.', False)


In [28]:
def run_adjudication(
    dataframe,
    checkpoint_name
):

    checkpoint_path = (
        OUTPUT_DIR / checkpoint_name
    )

    if checkpoint_path.exists():

        output = pd.read_pickle(
            checkpoint_path
        )

        print(
            "Resuming checkpoint:",
            checkpoint_name
        )

    else:

        output = dataframe.copy().reset_index(
            drop=True
        )

        output["stage_c_predictions"] = None
        output["stage_c_reason"] = None
        output["fallback_used"] = False

    for row_number, row in output.iterrows():

        if output.at[
            row_number,
            "stage_c_predictions"
        ] is not None:
            continue

        predictions, reason, fallback_used = (
            adjudicate(row)
        )

        output.at[
            row_number,
            "stage_c_predictions"
        ] = predictions

        output.at[
            row_number,
            "stage_c_reason"
        ] = reason

        output.at[
            row_number,
            "fallback_used"
        ] = fallback_used

        if (row_number + 1) % 10 == 0:

            output.to_pickle(
                checkpoint_path
            )

            print(
                f"Saved {row_number + 1} "
                f"of {len(output)}"
            )

    output.to_pickle(checkpoint_path)

    return output

In [29]:
train_llm_input = run_adjudication(
    train_llm_input,
    "stage_c_train_checkpoint.pkl"
)

Saved 10 of 100
Saved 20 of 100
Saved 30 of 100
Saved 40 of 100
Saved 50 of 100
Saved 60 of 100
Saved 70 of 100
Saved 80 of 100
Saved 90 of 100
Saved 100 of 100


In [30]:
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer


def evaluate_predictions(
    dataframe,
    prediction_column
):

    label_binarizer = MultiLabelBinarizer(
        classes=RISK_IDS
    )

    true_matrix = label_binarizer.fit_transform(
        dataframe["true_risks"]
    )

    predicted_matrix = label_binarizer.transform(
        dataframe[prediction_column]
    )

    print(
        classification_report(
            true_matrix,
            predicted_matrix,
            target_names=RISK_IDS,
            zero_division=0
        )
    )

    return {
        "micro_f1": f1_score(
            true_matrix,
            predicted_matrix,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            true_matrix,
            predicted_matrix,
            average="macro",
            zero_division=0
        ),
        "exact_match": np.mean(
            np.all(
                true_matrix == predicted_matrix,
                axis=1
            )
        )
    }

In [31]:
train_evaluation = train_llm_input.merge(
    train_poc[
        ["id", "true_risks"]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)

train_sample_metrics = evaluate_predictions(
    train_evaluation,
    "stage_c_predictions"
)

train_sample_metrics

                                         precision    recall  f1-score   support

                     weather_disruption       0.75      0.97      0.84        39
                       natural_disaster       0.67      0.67      0.67         3
            port_operational_disruption       0.84      0.84      0.84        61
                           port_closure       0.57      0.86      0.69        14
                labor_strike_disruption       0.68      1.00      0.81        13
maritime_security_navigation_disruption       0.57      0.53      0.55        15

                              micro avg       0.73      0.86      0.79       145
                              macro avg       0.68      0.81      0.73       145
                           weighted avg       0.74      0.86      0.79       145
                            samples avg       0.75      0.83      0.77       145



{'micro_f1': 0.7898089171974523,
 'macro_f1': 0.7328525180878206,
 'exact_match': np.float64(0.57)}

In [32]:
test_llm_input = run_adjudication(
    test_llm_input,
    "stage_c_test_checkpoint.pkl"
)

Saved 10 of 194
Saved 20 of 194
Saved 30 of 194
Saved 40 of 194
Saved 50 of 194
Saved 60 of 194
Saved 70 of 194
Saved 80 of 194
Saved 90 of 194
Saved 100 of 194
Saved 110 of 194
Saved 120 of 194
Saved 130 of 194
Saved 140 of 194
Saved 150 of 194
Saved 160 of 194
Saved 170 of 194
Saved 180 of 194
Saved 190 of 194


In [33]:
test_llm_input["retrieved_ids"] = (
    test_llm_input["retrieved_examples"].apply(
        lambda examples: [
            example["id"]
            for example in examples
        ]
    )
)


stage_c_results = test_llm_input[
    [
        "id",
        "stage_c_predictions",
        "stage_c_reason",
        "fallback_used",
        "retrieved_ids"
    ]
]


test_final = test_input.merge(
    stage_c_results,
    on="id",
    how="left",
    validate="one_to_one"
)


test_final["final_predictions"] = (
    test_final.apply(
        lambda row:
            row["stage_c_predictions"]
            if row["route_to_stage_c"]
            else row["accepted_predictions"],
        axis=1
    )
)


test_final["fallback_used"] = (
    test_final["fallback_used"]
    .fillna(False)
    .astype(bool)
)

/tmp/ipykernel_1127/3813298117.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [34]:
test_truth = pd.read_csv(
    DATA_DIR / "test_poc.csv",
    usecols=["id", "true_risks"]
)

test_truth["true_risks"] = (
    test_truth["true_risks"].apply(
        ast.literal_eval
    )
)


test_evaluation = test_final.merge(
    test_truth,
    on="id",
    how="left",
    validate="one_to_one"
)


full_test_metrics = evaluate_predictions(
    test_evaluation,
    "final_predictions"
)

routed_test_metrics = evaluate_predictions(
    test_evaluation[
        test_evaluation["route_to_stage_c"]
    ],
    "final_predictions"
)

                                         precision    recall  f1-score   support

                     weather_disruption       0.97      0.98      0.97       341
                       natural_disaster       0.91      0.91      0.91        35
            port_operational_disruption       0.85      0.89      0.87       351
                           port_closure       0.51      0.59      0.54        85
                labor_strike_disruption       0.98      1.00      0.99       179
maritime_security_navigation_disruption       0.73      0.74      0.74        97

                              micro avg       0.87      0.90      0.88      1088
                              macro avg       0.83      0.85      0.84      1088
                           weighted avg       0.87      0.90      0.89      1088
                            samples avg       0.91      0.93      0.91      1088

                                         precision    recall  f1-score   support

                     wea

In [35]:
metrics = pd.DataFrame(
    [
        train_sample_metrics,
        full_test_metrics,
        routed_test_metrics
    ],
    index=[
        "training_sample",
        "full_test_pipeline",
        "routed_test_only"
    ]
)


train_evaluation["retrieved_ids"] = (
    train_evaluation["retrieved_examples"].apply(
        lambda examples: [
            example["id"]
            for example in examples
        ]
    )
)


train_export = train_evaluation.drop(
    columns=["retrieved_examples"]
).copy()

test_export = test_final.copy()


json_columns = [
    "stage_a_predictions",
    "stage_b_predictions",
    "stage_b_probabilities",
    "accepted_predictions",
    "stage_c_predictions",
    "retrieved_ids",
    "final_predictions",
    "true_risks"
]


for dataframe in [train_export, test_export]:

    for column in json_columns:

        if column in dataframe.columns:

            dataframe[column] = dataframe[
                column
            ].apply(
                lambda value: json.dumps(
                    value,
                    ensure_ascii=False
                )
                if isinstance(
                    value,
                    (list, dict)
                )
                else value
            )

In [36]:
train_export.to_csv(
    OUTPUT_DIR /
    "stage_c_train_sample_predictions.csv",
    index=False
)

test_export.to_csv(
    OUTPUT_DIR /
    "stage_c_test_predictions.csv",
    index=False
)

metrics.to_csv(
    OUTPUT_DIR /
    "stage_c_metrics.csv",
    index_label="evaluation"
)

print("Stage C outputs saved.")

Stage C outputs saved.


In [37]:
train_results = pd.read_csv(
    OUTPUT_DIR /
    "stage_c_train_sample_predictions.csv"
)

test_results = pd.read_csv(
    OUTPUT_DIR /
    "stage_c_test_predictions.csv"
)

metrics = pd.read_csv(
    OUTPUT_DIR /
    "stage_c_metrics.csv"
)


assert len(train_results) == 100
assert len(test_results) == 819

assert test_results[
    "final_predictions"
].notna().all()


test_results["final_predictions"] = (
    test_results["final_predictions"].apply(
        json.loads
    )
)


route_mask = (
    test_results["route_to_stage_c"]
    .astype(str)
    .str.lower()
    .eq("true")
)

fallback_mask = (
    test_results["fallback_used"]
    .astype(str)
    .str.lower()
    .eq("true")
)


valid_risks = set(RISK_IDS)

invalid_count = (
    test_results["final_predictions"].apply(
        lambda predictions:
            not set(predictions).issubset(
                valid_risks
            )
    )
).sum()

empty_count = (
    test_results["final_predictions"]
    .apply(len)
    .eq(0)
    .sum()
)


assert route_mask.sum() == 194
assert invalid_count == 0


print("Training records:", len(train_results))
print("Test records:", len(test_results))
print("Stage C records:", route_mask.sum())
print("Fallbacks used:", fallback_mask.sum())
print("Empty final predictions:", empty_count)

display(metrics)

Training records: 100
Test records: 819
Stage C records: 194
Fallbacks used: 0
Empty final predictions: 0


,evaluation,micro_f1,macro_f1,exact_match
0,training_sample,0.789809,0.732853,0.570000
1,full_test_pipeline,0.884477,0.838277,0.772894
2,routed_test_only,0.825789,0.758254,0.541237
